# Ask the same SQL question of different sources
**Count January 2025 departures from Barcelona using a Parquet file, a Delta table and public Blob Storage.** Then explore optional shortcut, private Blob and Excel readers.
<br>Use Pure Python, attach `lh_pond`, and run `0. Setup` with `DATA_SCOPE = "one_year"`, including the Delta step. Run the connection cell first; each source defines its own path.

## Connect DuckDB: one question, several sources
**How many flights departed Barcelona in January 2025?** Keep the SELECT and filter; change the reader and source in FROM.
<br>This cell opens the connection used by the examples. The January source counts should match.

In [ ]:
import duckdb
con = duckdb.connect()

## 1. A file in the Lakehouse
Use the familiar Lakehouse **Files** path.
<br>`read_parquet` reads January’s file; `LEBL` selects Barcelona departures.

In [ ]:
MOUNT = "/lakehouse/default/Files/flights_eu/one_year/flight_list_202501.parquet"

con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM read_parquet('{MOUNT}')
    WHERE adep = 'LEBL'
""").show()

## 2. A Delta table in the Lakehouse
Setup created **`dbo.flights_202501`** from the January file.
<br>`delta_scan` uses the transaction log to identify the table’s current data files.
**INSTALL** makes the extension available. **LOAD** enables it for this connection.
<br>This query reads Delta storage directly; it does not use the SQL endpoint.

In [ ]:
TABLE = "/lakehouse/default/Tables/dbo/flights_202501"

con.execute("INSTALL delta")
con.execute("LOAD delta")

con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM delta_scan('{TABLE}')
    WHERE adep = 'LEBL'
""").show()

## 3. A public file in Azure Blob Storage
The **`httpfs` extension** reads Parquet over HTTPS.
<br>Upload the January 2025 flight Parquet file downloaded by `0. Setup` to your own public Blob container.
<br>Set `PUBLIC_BLOB` to its HTTPS URL, then set `RUN_PUBLIC_BLOB = True`. Leave it `False` to skip.
<br>Use the same dataset to preserve the expected columns and Barcelona flight count.


In [ ]:
RUN_PUBLIC_BLOB = False
PUBLIC_BLOB = "https://YOUR-STORAGE-ACCOUNT.blob.core.windows.net/YOUR-CONTAINER/flight_list_202501.parquet"

if RUN_PUBLIC_BLOB:
    if "YOUR-" in PUBLIC_BLOB:
        raise ValueError("Set PUBLIC_BLOB to your own public Parquet URL before enabling this example.")

    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{PUBLIC_BLOB}')
        WHERE adep = 'LEBL'
    """).show()
else:
    print("Public Blob example skipped. Set your URL, then RUN_PUBLIC_BLOB = True.")


## 4. Optional: read the Blob through a OneLake shortcut
**Reach the same public Blob file through a Lakehouse Files path.** Create `blob_flights` pointing to your own Blob container containing the January flight file first.
<br>Set `RUN_SHORTCUT = True` after creating it; leave `False` to skip.

In [ ]:
RUN_SHORTCUT = False

if RUN_SHORTCUT:
    SHORTCUT = "/lakehouse/default/Files/blob_flights/flight_list_202501.parquet"

    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{SHORTCUT}')
        WHERE adep = 'LEBL'
    """).show()
else:
    print("Shortcut example skipped. Create blob_flights, then set RUN_SHORTCUT = True.")

## Optional: read a private Blob with a Key Vault credential
**Key Vault supplies a SAS token; DuckDB's Azure extension reads the private file.** Configure your vault, secret and Blob address before enabling this example.
<br>Set `RUN_PRIVATE_BLOB = True` only with secret access and a valid SAS. Leave `False` to skip; keep the token out of output.

In [ ]:
RUN_PRIVATE_BLOB = False

if RUN_PRIVATE_BLOB:
    from notebookutils import credentials

    PRIVATE_BLOB = "azure://YOUR-CONTAINER/flight_list_202501.parquet"

    con.execute("INSTALL azure")
    con.execute("LOAD azure")

    # Retrieve the SAS token from Key Vault.
    sas_token = credentials.getSecret(
        "https://YOUR-KEY-VAULT.vault.azure.net/",
        "storage-sas-token",
    ).strip().lstrip("?")

    # Give DuckDB the credential for this storage account.
    con.execute(f"""
        CREATE OR REPLACE SECRET private_flights (
            TYPE azure,
            CONNECTION_STRING
                'BlobEndpoint=https://YOUR-STORAGE-ACCOUNT.blob.core.windows.net/;SharedAccessSignature={sas_token}'
        )
    """)

    # Query the private Parquet file and display the result.
    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{PRIVATE_BLOB}')
        WHERE adep = 'LEBL'
    """).show()

else:
    print("Private Blob example skipped. Set RUN_PRIVATE_BLOB = True after configuring your own resources.")

## Optional: recover the same total from Excel
**Read the workbook created by setup section 7 and sum its `flights` column.** It contains January Barcelona counts by destination, rather than individual flight rows.
<br>Set `RUN_EXCEL = True` after creating it. `read_xlsx` handles `.xlsx`, not `.xls`; cast the total to BIGINT because Excel numbers are read as doubles.

In [ ]:
RUN_EXCEL = True

if RUN_EXCEL:
    from pathlib import Path

    EXCEL = "/lakehouse/default/Files/flights_eu/excel/barcelona_departures_202501.xlsx"

    if Path(EXCEL).exists():
        con.execute("INSTALL excel")
        con.execute("LOAD excel")

        con.sql(f"""
            SELECT CAST(sum(flights) AS BIGINT) AS flights_from_barcelona
            FROM read_xlsx('{EXCEL}')
        """).show()
    else:
        print("Workbook not found. Run 0. Setup section 7 to create it.")
else:
    print("Excel example skipped. Set RUN_EXCEL = True to run it.")

## What changed?
**The SQL question stayed the same. The reader, path and access method changed.**
| Source | Reader / access |
| --- | --- |
| Lakehouse Parquet | `read_parquet` |
| Lakehouse Delta | `delta_scan` reads the transaction log |
| Public Blob | `read_parquet` over HTTPS with `httpfs` |
| OneLake shortcut | `read_parquet` through the Lakehouse Files path |
| Private Blob | `read_parquet` with Azure extension and Key Vault credential |
| Excel | `read_xlsx`; sum the pre-aggregated flight counts |
Create a Lakehouse copy when the workflow needs one.